# Mini-Project 3 — A Framework RAG with LangChain
### IT7075: Applied AI for Cybersecurity · LangChain

In Mini-Project 2 you built RAG **by hand**: your own chunker, your own Chroma calls, your own
prompt assembly, your own retrieve-then-generate function. Now you build the same thing **with
LangChain**, and the point of the exercise is to see exactly which parts the framework takes over.

Five pieces are left for you to write, one per stage of the pipeline in the langchain_rag notebook:

| | Stage | You write |
|---|---|---|
| **TODO 1** | load the documents and split them | 3 lines |
| **TODO 2** | build the Chroma vector store | 1 line |
| **TODO 3** | turn it into a retriever (`k` + score threshold) | 2 lines |
| **TODO 4** | write the prompt template | 2 lines |
| **TODO 5** | wire the LCEL chain together with `\|` | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs offline.** Embeddings are the local `all-MiniLM-L6-v2` model, so retrieval never needs an
> API key. If you have a working `OPENAI_API_KEY` the last stage of the chain calls a real model;
> if you don't — or it is out of credit — the notebook substitutes an offline answer and keeps
> going. Everything you are graded on works either way.

## Part 1 — Load and split the documents  ·  **TODO 1**

LangChain's loaders return **`Document` objects** (text plus metadata), not plain strings — that is
the first difference from Mini-Project 2. The splitter then works on documents and hands back more
documents, carrying the metadata along for you.

> **Expected output:** `1 document -> 10 chunks`.

In [1]:
# If needed: !pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-openai
import json, os, warnings
import pandas as pd

# Keep this notebook on the CPU. The models here are tiny (milliseconds either way),
# and this avoids the "CUDA error: no kernel image is available" failure -- or a very
# long hang while torch initialises -- on a machine whose PyTorch build does not match
# its GPU. This must run BEFORE anything imports torch.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

# Two noisy-but-harmless warnings, silenced so the output stays readable:
#  * langchain-community is being retired (TextLoader still works and is what the langchain_rag notebook uses)
#  * LangChain warns when a relevance score falls outside 0-1. With these embeddings an
#    UNRELATED question scores slightly below 0 -- which is the behaviour we want. See Part 3.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="Relevance scores must be between")

# LangChain also logs "No relevant docs were retrieved..." every time the threshold
# rejects everything. That is the behaviour we are measuring, and the cells below
# already report it as "retrieved 0 chunk(s)", so we log it once instead of 20 times.
import logging
logging.getLogger("langchain_core.vectorstores.base").setLevel(logging.ERROR)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# ---- the knowledge base and question set that came with the course -------
def course_path(name, folders=("course_kb/", "./")):
    for f in folders:
        if os.path.exists(f + name):
            return f + name
    return name

KB_FILE = course_path("ai_framework_kb.md")
GOLDSET_FILE = ("framework_goldset.json" if os.path.exists("framework_goldset.json")
                else "../framework_goldset.json")


###################  TODO 1 — load the file and split it into chunks  ###################
# WHAT: load KB_FILE into `documents`, then split it into `chunks`.
#
# HOW (the "Load & Split" cell in the langchain_rag notebook):
#   * loader    = TextLoader(KB_FILE, encoding="utf-8")
#   * documents = loader.load()
#         -> a LIST of Document objects. A whole .md file loads as ONE document,
#            so len(documents) is 1. The text is in documents[0].page_content.
#   * splitter  = CharacterTextSplitter(chunk_size=900, chunk_overlap=150,
#                                       separator="\n")      <- given below
#   * chunks    = splitter.split_documents(documents)
#         -> note split_documentS: it takes Documents and returns Documents,
#            keeping the metadata. (split_text() would take a plain string.)
#
# WHY separator="\n": the splitter tries to break on that separator first, so
# chunks end at line boundaries instead of mid-sentence like in Mini-Project 2.
################

loader = TextLoader(KB_FILE, encoding="utf-8")
documents = loader.load()
splitter = CharacterTextSplitter(chunk_size=900, chunk_overlap=150, separator="\n")
chunks = splitter.split_documents(documents)

print(len(documents), "document ->", len(chunks), "chunks")
print("\nFirst chunk:\n", chunks[0].page_content[:200], "...")
print("\nIts metadata:", chunks[0].metadata)

1 document -> 10 chunks

First chunk:
 # NIST AI Risk Management Framework (AI RMF)
The NIST AI Risk Management Framework (AI RMF 1.0) is a voluntary framework to help organizations manage risks across the AI lifecycle. It is organized aro ...

Its metadata: {'source': 'course_kb/ai_framework_kb.md'}


## Part 2 — Build the vector store  ·  **TODO 2**

In Mini-Project 2 this took three steps: embed the chunks, create a collection, then add ids,
documents, embeddings and metadata. LangChain does all of it in **one call** — `Chroma.from_documents`
embeds and stores in a single line. That is the framework's whole pitch.

We use the **local** embedding model, so no API key is needed.

> **Expected output:** `stored 10 chunks`.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# The offline embedding model the langchain_rag notebook names in its comment.
# Swap in OpenAIEmbeddings(model="text-embedding-3-small") if you have a working key.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


###################  TODO 2 — put the chunks into Chroma  ###################
# WHAT: build the vector store in ONE line.
#
# HOW (the "Embed & Store" cell in the langchain_rag notebook):
#       Chroma.from_documents(<the chunks>, <the embeddings>)
#
# That single call embeds every chunk with `embeddings` and stores the vectors,
# the text, and the metadata. Compare it with Mini-Project 2, where you called
# embed() yourself and then collection.add(ids=..., documents=..., embeddings=...,
# metadatas=...). Same work; the framework hides it.
#
# NOTE: we leave out persist_directory on purpose, so the store lives in memory and
#       re-running this cell can never mix new chunks with stale ones.
################

db = Chroma.from_documents(chunks, embeddings)

print("stored", db._collection.count(), "chunks")

W1102 13:09:18.080000 29572 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

stored 10 chunks


## Part 3 — Make a retriever  ·  **TODO 3**

A **retriever** is a vector store wrapped in a standard interface, so anything downstream can just
call `.invoke(question)`. It is also where your two Mini-Project 2 knobs reappear under new names:

| Mini-Project 2 (by hand) | LangChain |
|---|---|
| `k` = how many chunks | `search_kwargs={"k": ...}` |
| `max_distance` — keep hits **below** it | `score_threshold` — keep hits **above** it |

Note the flip: a **distance** is small when things are similar, but LangChain's **relevance score**
is large when things are similar. Same idea, opposite direction.

> **Expected output:** 3 chunks for the NIST question, **0** for "the capital of France" — the
> refusal you built by hand in Mini-Project 2, now handled by the retriever.

In [3]:
###################  TODO 3 — wrap the store in a retriever  ###################
# WHAT: create `retriever` from `db`, keeping at most K chunks and only those
#       scoring above THRESHOLD.
#
# HOW (the "Make a Retriever" cell in the langchain_rag notebook):
#       db.as_retriever(
#           search_type="similarity_score_threshold",
#           search_kwargs={"k": K, "score_threshold": THRESHOLD})
#
#   * search_type is the STRING "similarity_score_threshold"
#   * search_kwargs is a DICT with two keys: "k" and "score_threshold"
#
#   * search_type="similarity_score_threshold" is what makes score_threshold apply.
#     With the default search_type="similarity" the threshold is IGNORED and you
#     always get K chunks back -- the bug that made your Mini-Project 2 bot answer
#     questions its documents could not support.
#   * relevance score: HIGHER = more similar (the opposite of a distance).
#   * use the constants K and THRESHOLD below, not literal numbers -- Part 6
#     changes them.
################

K = 3
THRESHOLD = 0.3          # the langchain_rag value. Part 6 asks whether it is right.

retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": K, "score_threshold": THRESHOLD})


# ---- given: see what comes back, and what does not ----------------------
for q in ["What are the four core functions of the NIST AI Risk Management Framework?",
          "What is the capital of France?"]:
    got = retriever.invoke(q)
    print("Q:", q)
    print("   retrieved", len(got), "chunk(s)")
    for d in got:
        print("     -", d.page_content[:70].strip().replace("\n", " "), "...")
    print()

Q: What are the four core functions of the NIST AI Risk Management Framework?
   retrieved 3 chunk(s)
     - # NIST AI Risk Management Framework (AI RMF) The NIST AI Risk Manageme ...
     - # NIST Generative AI Profile The NIST AI RMF Generative AI Profile (NI ...
     - # EU AI Act The EU Artificial Intelligence Act is the first comprehens ...

Q: What is the capital of France?
   retrieved 0 chunk(s)



## Part 4 — The prompt template  ·  **TODO 4**

A `ChatPromptTemplate` is the reusable prompt from the langchain_basics notebook: you write it once with `{}`
placeholders, and the chain fills them in on every question. This replaces the string `.format()`
you did by hand in Mini-Project 2.

> **Expected output:** the filled-in prompt, showing your context in the system message and the
> question in the human message.

In [4]:
###################  TODO 4 — write the prompt template  ###################
# WHAT: build `rag_prompt` with a system message and a human message.
#
# HOW (prompt templates in langchain_basics + the chain cell in langchain_rag):
#       rag_prompt = ChatPromptTemplate.from_messages([
#           ("system", "....{context}...."),
#           ("human", "{question}"),
#       ])
#
# The system message must:
#   * tell the model to answer using ONLY the context
#   * contain the placeholder {context}
#   * say to start the reply with [RAG] when the answer is in the context, and with
#     [LLM] plus "I don't know" when it is not -- the same tagging as Mini-Project 2
# The human message is just the placeholder {question}.
#
# WATCH OUT: the two placeholder names must be exactly `context` and `question`,
#            because that is what the chain in Part 5 feeds in.
################

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a cybersecurity assistant. Answer the question using ONLY the context below. "
               "If the answer is in the context, start your reply with [RAG]. If the context is empty or "
               "does not contain the answer, start with [LLM] and say I don't know.\n\n# Context\n{context}"),
    ("human", "{question}"),
])

# ---- given: check what the filled-in prompt looks like ------------------
preview = rag_prompt.invoke({"context": "(pretend context here)", "question": "test question"})
for m in preview.to_messages():
    print("[%s] %s\n" % (m.type, m.content[:220]))

[system] You are a cybersecurity assistant. Answer the question using ONLY the context below. If the answer is in the context, start your reply with [RAG]. If the context is empty or does not contain the answer, start with [LLM] 

[human] test question



## Part 5 — Wire the chain with `|`  ·  **TODO 5**

This is the part that has no equivalent in Mini-Project 2. There you wrote an `ask()` function that
called retrieve, then built the context, then called generate. In LangChain you **declare** that
flow with the pipe operator and get a single runnable object back:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}  |  prompt  |  model  |  parser
```

Read it left to right: the dict runs the retriever **and** passes the raw question through, the
prompt fills both placeholders, the model answers, and the parser turns the reply into a string.

> **Expected output:** the NIST question answered and tagged `[RAG]`; "the capital of France"
> refused and tagged `[LLM]`, because the retriever returned nothing for it.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


def format_docs(docs):
    """GIVEN: turn the retrieved Documents into one block of text for {context}."""
    return "\n\n".join(d.page_content for d in docs)


def offline_answer(prompt_value):
    """GIVEN: stands in for the model when no LLM is available (a RunnableLambda,
    exactly like the ones in langchain_basics). It reads the context out of the filled-in
    prompt and quotes it, so the chain still produces a tagged answer."""
    context = prompt_value.to_messages()[0].content.split("# Context", 1)[-1].strip()
    if context == "":
        return "[LLM] I don't know -- the knowledge base does not cover this. (offline mode)"
    return "[RAG] (offline mode) From the retrieved context: " + context[:220]


def build_model():
    """GIVEN: a real chat model if a working key is available, else the offline stand-in."""
    if os.getenv("OPENAI_API_KEY"):
        try:
            from langchain_openai import ChatOpenAI
            m = ChatOpenAI(model="gpt-4o-mini", temperature=0)
            m.invoke("ping")                       # fail fast if the key has no credit
            print("using ChatOpenAI(gpt-4o-mini)")
            return m | StrOutputParser()
        except Exception as e:
            print("(LLM unavailable - %s. Using offline mode.)" % type(e).__name__)
    else:
        print("(no OPENAI_API_KEY - using offline mode.)")
    return RunnableLambda(offline_answer)


model = build_model()


###################  TODO 5 — build the LCEL chain  ###################
# WHAT: assemble `rag_chain` from the four pieces you already have, using `|`.
#
# HOW (the "Build the RAG Chain (LCEL)" cell in the langchain_rag notebook):
#       rag_chain = (
#           {"context": <retriever piped into format_docs>,
#            "question": <pass the question through untouched>}
#           | <your prompt>
#           | <the model>
#       )
#
#   * the DICT is the interesting bit: both entries receive the question you invoke
#     the chain with. "context" sends it through the retriever and then through
#     format_docs; "question" passes it along untouched (RunnablePassthrough()).
#     The result is {"context": "...", "question": "..."} -- exactly the two
#     placeholders your prompt needs.
#   * `model` here already includes the output parser, so you do not add one.
#
# THEN: invoke it with just a question string -- rag_chain.invoke("...?")
################

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
)

# ---- given: one question the KB answers, one it does not ----------------
for q in ["What are the four core functions of the NIST AI Risk Management Framework?",
          "What is the capital of France?"]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q)[:300], "\n")

(LLM unavailable - ModuleNotFoundError. Using offline mode.)
Q: What are the four core functions of the NIST AI Risk Management Framework?
A: [RAG] (offline mode) From the retrieved context: # NIST AI Risk Management Framework (AI RMF)
The NIST AI Risk Management Framework (AI RMF 1.0) is a voluntary framework to help organizations manage risks across the AI lifecycle. It is organized around four core functi 

Q: What is the capital of France?
A: [LLM] I don't know -- the knowledge base does not cover this. (offline mode) 



## Part 6 — Ask the bot, and measure it  *(no TODO — run it and read it)*

`framework_goldset.json` holds **8 questions** about the provided knowledge base: **6 it answers**
(each with a phrase that proves it) and **2 it does not**. The two counts are the same ones you
used in Mini-Project 2:

- **found** — of the 6, how many retrieved a chunk containing the evidence?
- **refused** — of the 2, how many correctly retrieved *nothing*?

> **Expected output at the langchain_rag settings (`k=3`, threshold 0.3): found 5/6, refused 1/2.**
> Two things go wrong at once, and neither is a bug in your code:
> **Q2** (MITRE ATLAS) finds nothing, and **Q8** — which the knowledge base cannot answer —
> gets context anyway. Part 7 shows why, and why you cannot fix both by tuning.

In [6]:
GOLDSET = json.load(open(GOLDSET_FILE))["questions"]
N_ANS = sum(q["answerable"] for q in GOLDSET)
N_UNANS = len(GOLDSET) - N_ANS


def score(retriever, goldset):
    """GIVEN: count found / refused, the same two numbers as Mini-Project 2."""
    found, refused, detail = 0, 0, []
    for q in goldset:
        docs = retriever.invoke(q["question"])
        text = " ".join(d.page_content for d in docs).lower()
        hit = any(p.lower() in text for p in q["evidence"])
        if q["answerable"]:
            found += 1 if hit else 0
        else:
            refused += 1 if len(docs) == 0 else 0
        detail.append({"qid": q["qid"], "answerable": q["answerable"],
                       "chunks": len(docs), "evidence_found": hit if q["answerable"] else "-"})
    return found, refused, detail


found, refused, detail = score(retriever, GOLDSET)
print("found   %d/%d" % (found, N_ANS))
print("refused %d/%d\n" % (refused, N_UNANS))
print(pd.DataFrame(detail).to_string(index=False))

found   5/6
refused 1/2

qid  answerable  chunks evidence_found
 Q1        True       3           True
 Q2        True       0          False
 Q3        True       2           True
 Q4        True       3           True
 Q5        True       3           True
 Q6        True       1           True
 Q7       False       0              -
 Q8       False       3              -


## Part 7 — Tune the threshold  *(no TODO)*

Sweep the threshold and watch the two counts fight each other.

> **Expected output:**
>
> | threshold | found | refused | what changed |
> |---|---:|---:|---|
> | 0.1 | **6/6** | 1/2 | finds everything, but Q8 still gets context |
> | 0.2 | **6/6** | 1/2 | same |
> | **0.3** *(lecture default)* | 5/6 | 1/2 | **worst of both** — loses Q2 *and* still leaks Q8 |
> | 0.4 | 5/6 | 1/2 | |
> | 0.5 | 5/6 | 1/2 | |
> | 0.6 | 4/6 | **2/2** | finally refuses Q8 — by giving up Q2 and Q3 too |
>
> **Read that table carefully: there is no good row.** No threshold gets 6/6 *and* 2/2, and the
> last cell shows you why — Q8, which your documents **cannot** answer, scores **0.505**, while
> Q2, which they answer perfectly well, scores only **0.258**. The wrong answer looks *more*
> relevant than the right one, so no single cut-off can separate them.
>
> That is the honest finding of this project, and it is worth more than a tidy result: **tuning
> cannot fix a ranking that is already wrong.** Fixing it would need better chunking, a better
> embedding model, or a reranking step — not a different number.
>
> Mini-Project 2 showed you the other half of this trade-off: there a *loose* threshold let the
> bot answer what it shouldn't. Both failures are silent, and only a question set catches either.

In [7]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    r = db.as_retriever(search_type="similarity_score_threshold",
                        search_kwargs={"k": K, "score_threshold": t})
    f, ref, _ = score(r, GOLDSET)
    rows.append({"threshold": t, "found": "%d/%d" % (f, N_ANS),
                 "refused": "%d/%d" % (ref, N_UNANS)})

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("results.csv", index=False)
print("\nwrote results.csv")

# Why is there no good row? Compare the best score for a question the KB ANSWERS
# with the best score for one it CANNOT answer.
print("\ntop-1 relevance score per question (higher = looks more relevant):")
for q in GOLDSET:
    s = db.similarity_search_with_relevance_scores(q["question"], k=1)[0][1]
    print("   %-4s %-7s %+.3f  %s" % (q["qid"], "answer" if q["answerable"] else "ABSENT",
                                      s, q["question"][:52]))
print("\nQ8 is not in the knowledge base yet outranks Q2, which is. No cut-off fixes that.")

 threshold found refused
       0.1   6/6     1/2
       0.2   6/6     1/2
       0.3   5/6     1/2
       0.4   5/6     1/2
       0.5   5/6     1/2
       0.6   4/6     2/2

wrote results.csv

top-1 relevance score per question (higher = looks more relevant):
   Q1   answer  +0.640  What are the four core functions of the NIST AI Risk
   Q2   answer  +0.258  What is MITRE ATLAS used for?
   Q3   answer  +0.548  Name three risks from the OWASP Top 10 for LLM Appli
   Q4   answer  +0.696  What obligations does the EU AI Act place on high-ri
   Q5   answer  +0.656  What are the core elements of Google's Secure AI Fra
   Q6   answer  +0.623  What is the MAESTRO threat modeling framework used f
   Q7   ABSENT  -0.283  What is the capital of France?
   Q8   ABSENT  +0.505  What are the five core functions of the NIST Cyberse

Q8 is not in the knowledge base yet outranks Q2, which is. No cut-off fixes that.


## Part 8: rebuild your own RAG with LangChain

Everything above used the provided knowledge base, and it was practice. Now build the same system you
built in the RAG module, this time with LangChain, using the same data and the same questions.

Copy your `my_kb/` folder and your `goldset.json` from the RAG module into this project folder. Do not
change the documents or the questions: the comparison in your report depends on both systems answering
exactly the same eight questions over exactly the same text.

No code is provided from here on. Add cells below that:

1. Load your documents with a LangChain loader and split them with a text splitter.
2. Build a vector store from the chunks and wrap it in a retriever.
3. Write a prompt template and wire the chain with the pipe operator.
4. Answer your eight questions and print the transcript, including one the documents cannot answer.
5. Score the same measurements you used in the RAG module, vary one setting at a time, and save the
   results as `results_langchain.csv`.

Note as you go which of your RAG-module functions each LangChain object replaced. That list is the
heart of your report.

In [8]:
###################  YOUR RAG, REBUILT WITH LANGCHAIN  ###################
import glob, urllib.request
from langchain_ollama import ChatOllama

# oversized chunks are counted in the results table instead of logged one by one
logging.getLogger("langchain_text_splitters.base").setLevel(logging.ERROR)

# ---- load: one TextLoader per file -> Document objects with a "source" in metadata
MY_FILES = sorted(f for f in glob.glob("my_kb/*.md") if os.path.basename(f) != "SOURCES.md")
my_docs = []
for f in MY_FILES:
    my_docs += TextLoader(f, encoding="utf-8").load()
print("Loaded %d documents, %d characters" % (len(my_docs), sum(len(d.page_content) for d in my_docs)))

MY_GOLDSET = json.load(open("goldset.json", encoding="utf-8"))["questions"]
all_text = "\n".join(d.page_content for d in my_docs)
missing = [p for q in MY_GOLDSET for p in q["evidence"] if p not in all_text]
print("Loaded %d questions; evidence phrases missing from my_kb/: %d\n" % (len(MY_GOLDSET), len(missing)))
assert not missing, missing

# ---- split + store: same settings as the RAG module (600 / 100), cosine space
CHUNK_SIZE, OVERLAP, K, THRESHOLD = 600, 100, 4, 0.25     # 0.25 relevance == 0.75 cosine distance
_n = [0]

def build_store(chunk_size, overlap):
    splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap, separator="\n")
    pieces = splitter.split_documents(my_docs)
    _n[0] += 1
    store = Chroma.from_documents(pieces, embeddings, collection_name="my_kb_%d" % _n[0],
                                  collection_metadata={"hnsw:space": "cosine"},
                                  relevance_score_fn=lambda d: 1.0 - d)
    return store, pieces

my_db, my_chunks = build_store(CHUNK_SIZE, OVERLAP)
sizes = [len(c.page_content) for c in my_chunks]
print("Split into %d chunks (smallest %d, largest %d chars, %d over %d); stored %d"
      % (len(my_chunks), min(sizes), max(sizes), sum(s > CHUNK_SIZE for s in sizes), CHUNK_SIZE,
         my_db._collection.count()))

# ---- retriever: keep at most K chunks scoring at or above THRESHOLD
my_retriever = my_db.as_retriever(search_type="similarity_score_threshold",
                                  search_kwargs={"k": K, "score_threshold": THRESHOLD})

# ---- prompt template
my_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a compliance assistant for U.S. defense contractors. Answer the question using ONLY the "
     "context below, which contains excerpts from DFARS, CMMC (32 CFR 170) and ITAR regulations.\n"
     "Cite the source file for every claim, like [dfars_7012.md]. If the answer is in the context, start "
     "your reply with [RAG]. If the context is empty or does not contain the answer, say you do not "
     "know and start with [LLM] -- do not answer from memory.\n\n# Context\n{context}"),
    ("human", "{question}"),
])

def format_docs_with_source(docs):
    return "\n\n".join("[%d] (source: %s)\n%s" % (i, os.path.basename(d.metadata["source"]), d.page_content)
                       for i, d in enumerate(docs, start=1))

# ---- model: mistral:7b on loopback Ollama, else the offline stand-in from Part 5
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3)
        return True
    except OSError:
        return False

if ollama_up():
    my_model = ChatOllama(model="mistral:7b", base_url="http://127.0.0.1:11434",
                          temperature=0, num_ctx=4096) | StrOutputParser()
    print("Generator: mistral:7b via Ollama on 127.0.0.1:11434")
else:
    my_model = RunnableLambda(offline_answer)
    print("Generator: offline mode (Ollama not reachable)")

# ---- the chain
my_chain = (
    {"context": my_retriever | format_docs_with_source, "question": RunnablePassthrough()}
    | my_prompt
    | my_model
)


Loaded 5 documents, 9869 characters
Loaded 8 questions; evidence phrases missing from my_kb/: 0

Split into 22 chunks (smallest 61, largest 761 chars, 1 over 600); stored 22
Generator: mistral:7b via Ollama on 127.0.0.1:11434


### Answer your questions and measure the result

In [9]:
###################  ANSWER AND MEASURE  ###################
# ---- transcript: all eight questions
for q in MY_GOLDSET:
    shown = my_db.similarity_search_with_relevance_scores(q["question"], k=K, score_threshold=THRESHOLD)
    print("Q:", q["question"])
    if shown:
        for d, s in shown:
            print("   retrieved  score %.3f  [%s]" % (s, os.path.basename(d.metadata["source"])))
    else:
        print("   retrieved  nothing (no chunk scored %.2f or higher)" % THRESHOLD)
    print("   answer:", my_chain.invoke(q["question"]).strip().replace("\n", "\n           "), "\n")

# ---- experiments: one setting at a time from the RAG-module baseline
def run_config(experiment, chunk_size, overlap, k, threshold):
    store, pieces = build_store(chunk_size, overlap)          # fresh store for every configuration
    stored = store._collection.count()
    assert stored == len(pieces), "store holds %d of %d chunks" % (stored, len(pieces))
    r = store.as_retriever(search_type="similarity_score_threshold",
                           search_kwargs={"k": k, "score_threshold": threshold})
    f, ref, detail = score(r, MY_GOLDSET)
    store.delete_collection()
    missed = [d["qid"] for d in detail
              if (d["answerable"] and not d["evidence_found"]) or (not d["answerable"] and d["chunks"] > 0)]
    return {"experiment": experiment, "chunk_size": chunk_size, "overlap": overlap, "k": k,
            "score_threshold": threshold, "equiv_max_distance": round(1 - threshold, 2),
            "n_chunks": len(pieces), "max_chunk": max(len(c.page_content) for c in pieces),
            "n_oversized": sum(len(c.page_content) > chunk_size for c in pieces), "found": f, "found_of": N_ANS_MY, "refused": ref,
            "refused_of": N_UNANS_MY, "missed": " ".join(missed) or "-",
            "like_for_like": (chunk_size, overlap, k, threshold) == (600, 100, 4, 0.25)}

N_ANS_MY = sum(q["answerable"] for q in MY_GOLDSET)
N_UNANS_MY = len(MY_GOLDSET) - N_ANS_MY

rows = []
for cs in [200, 300, 400, 600, 800, 1000, 1500]:
    rows.append(run_config("chunk_size", cs, OVERLAP, K, THRESHOLD))
for k in [1, 2, 3, 4, 6, 8]:
    rows.append(run_config("k", CHUNK_SIZE, OVERLAP, k, THRESHOLD))
for t in [0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.1, 0.0]:
    rows.append(run_config("score_threshold", CHUNK_SIZE, OVERLAP, K, t))

results_lc = pd.DataFrame(rows)
results_lc.to_csv("results_langchain.csv", index=False)
print(results_lc.to_string(index=False))
print("\nwrote results_langchain.csv")

# ---- like-for-like comparison with the RAG module
mp3_path = "../../module04/rag_project/results.csv"
if os.path.exists(mp3_path):
    mp3 = pd.read_csv(mp3_path)
    base = mp3[(mp3.chunk_size == 600) & (mp3.k == 4) & (mp3.max_distance == 0.75)].iloc[0]
    lc = results_lc[results_lc.like_for_like].iloc[0]
    print("\nLike for like (chunk 600, overlap 100, k=4, distance 0.75 == score 0.25):")
    print("   RAG module : %d chunks, found %d/6, refused %d/2, missed %s"
          % (base.n_chunks, base.found, base.refused, base.missed))
    print("   LangChain  : %d chunks, found %d/6, refused %d/2, missed %s"
          % (lc.n_chunks, lc.found, lc.refused, lc.missed))

# ---- top chunk per question at the baseline
print("\nTop-1 chunk per question (relevance score; 1 - score = cosine distance):")
for q in MY_GOLDSET:
    d, s = my_db.similarity_search_with_relevance_scores(q["question"], k=1)[0]
    print("   %s %-5s score %.3f  dist %.3f  [%s]  %s..." % (
        q["qid"], "ans" if q["answerable"] else "unans", s, 1 - s,
        os.path.basename(d.metadata["source"]), d.page_content[:60].replace("\n", " ")))


Q: How quickly does a defense contractor have to report a cyber incident to the DoD?
   retrieved  score 0.663  [dfars_7012.md]
   retrieved  score 0.652  [dfars_7012.md]
   retrieved  score 0.578  [dfars_7012.md]
   retrieved  score 0.535  [dfars_7012.md]
   answer: [RAG] The defense contractor has to report a cyber incident to the DoD rapidly, which means within 72 hours of discovery, according to [dfars_7012.md]. 

Q: How long must a contractor keep images of affected systems after reporting a cyber incident?
   retrieved  score 0.782  [dfars_7012.md]
   retrieved  score 0.517  [dfars_7012.md]
   retrieved  score 0.491  [dfars_7012.md]
   retrieved  score 0.458  [dfars_7012.md]
   answer: [RAG] According to [dfars_7012.md], the contractor shall preserve and protect images of all known affected information systems for at least 90 days from the submission of the cyber incident report, as stated in [dfars_7012.md] (e). 

Q: Who performs a CMMC Level 3 certification assessment?
   retri